In [0]:
%sql
-- select everything
select * 
from default.loans

In [0]:
%sql
select *
from default.loans 
where loan_id is null 
      or customer_id is null 
      or institution_id is null 
      or loan_amount is null 
      or loan_status is null 
      or interest_rate is null 
      or loan_tenure_months is null 
      or application_date is null 
      or disbursement_date is null
      or maturity_date is null 
      or emi_amount is null 
      or purpose_of_loan is null

In [0]:
%sql
select *
from default.loans 
where loan_amount <=0 
      or emi_amount <= 0
      or interest_rate <= 0

In [0]:
%sql
select *
from default.loans 
where application_date >= disbursement_date 
      or application_date >= maturity_date 
      or disbursement_date >= maturity_date

In [0]:
%sql
select *  
from default.loans 
where loan_amount <= 0
      or interest_rate not between 5 and 20 
      or loan_tenure_months < 0

In [0]:
%sql
with xyz as 
(select loan_id, round(emi_amount) as emi_amount, 

round(loan_amount * (interest_rate/1200) * power((1 + interest_rate/1200) , loan_tenure_months) / 
(power((1 + interest_rate/1200), loan_tenure_months) - 1)) as emi_amount_calc

from default.loans)

select loan_id, emi_amount, emi_amount_calc
from xyz 
where abs(emi_amount - emi_amount_calc) > 2

In [0]:
%sql
-- create bronze table

create or replace table default.bronze_loans 
as

with silver_cte as
(select 
loan_id,
customer_id,
institution_id,
loan_amount,
loan_status,
interest_rate,
loan_tenure_months,
application_date,
disbursement_date,
maturity_date,
emi_amount,
purpose_of_loan
from default.loans)

select 
loan_id,
customer_id,
institution_id,
loan_amount,
loan_status,
interest_rate,
loan_tenure_months,
application_date,
disbursement_date,
maturity_date,
round(emi_amount) as emi_amount,
purpose_of_loan

from silver_cte
where loan_id is not null 
      and customer_id is not null 
      and institution_id is not null 
      and loan_amount is not null 
      and loan_status is not null 
      and interest_rate is not null 
      and loan_tenure_months is not null 
      and application_date is not null 
      and disbursement_date is not null
      and maturity_date is not null 
      and emi_amount is not null 
      and purpose_of_loan is not null
      and loan_amount > 0 
      and emi_amount > 0
      and interest_rate > 0
      and application_date < disbursement_date 
      and application_date < maturity_date 
      and disbursement_date < maturity_date
      and loan_amount > 0
      and interest_rate between 5 and 20 
      and loan_tenure_months > 0
      

In [0]:
%sql
select count(*) = (select count(*) from default.loans)
from default.bronze_loans